In [1]:
!pip install -q langchain-ollama langchain-core requests


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [3]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [4]:
print(multiply.invoke({'a':3, 'b':4}))

12


In [5]:
multiply.name

'multiply'

In [6]:
multiply.description

'Given 2 numbers a and b this tool returns their product'

In [7]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [8]:
# tool binding
llm = ChatOllama(model="llama3.1:8b")

In [9]:
llm_with_tools = llm.bind_tools([multiply])

In [10]:
query = HumanMessage(content='What is 3 multiplied by 4?')

In [11]:
messages=[query]

In [12]:
messages

[HumanMessage(content='What is 3 multiplied by 4?', additional_kwargs={}, response_metadata={})]

In [13]:
llm_with_tools.invoke('What is 3 multiplied by 4?').tool_calls #llm wont call it just suggest that this tool 'multiply' here can be used to solve this problem with the helkp of tool_call 

[{'name': 'multiply',
  'args': {'b': 4, 'a': 3},
  'id': '70158d0c-7546-44c8-8336-f980373b18b1',
  'type': 'tool_call'}]

In [14]:
result = llm_with_tools.invoke(messages)

In [15]:
messages = [query, result]

In [16]:
messages

[HumanMessage(content='What is 3 multiplied by 4?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-05-25T15:00:56.0691507Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2542173400, 'load_duration': 248867200, 'prompt_eval_count': 165, 'prompt_eval_duration': 100603200, 'eval_count': 22, 'eval_duration': 2150433900, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019e5fa7-329a-7382-b6b7-aa8ab5865cd9-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 4}, 'id': '35f9612b-49da-4cee-a176-f8a7d3559893', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 165, 'output_tokens': 22, 'total_tokens': 187})]

In [17]:
tool_result = multiply.invoke(result.tool_calls[0])

In [18]:
messages = [query, result, tool_result]

In [19]:
messages

[HumanMessage(content='What is 3 multiplied by 4?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-05-25T15:00:56.0691507Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2542173400, 'load_duration': 248867200, 'prompt_eval_count': 165, 'prompt_eval_duration': 100603200, 'eval_count': 22, 'eval_duration': 2150433900, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019e5fa7-329a-7382-b6b7-aa8ab5865cd9-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 4}, 'id': '35f9612b-49da-4cee-a176-f8a7d3559893', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 165, 'output_tokens': 22, 'total_tokens': 187}),
 ToolMessage(content='12', name='multiply', tool_call_id='35f9612b-49da-4cee-a176-f8a7d3559893')]

In [20]:
llm_with_tools.invoke(messages).content

'The result of multiplying 3 by 4 is 12.'

In [21]:
result.tool_calls[0]

{'name': 'multiply',
 'args': {'a': 3, 'b': 4},
 'id': '35f9612b-49da-4cee-a176-f8a7d3559893',
 'type': 'tool_call'}

In [22]:
#tool execution
multiply.invoke(result.tool_calls[0])

ToolMessage(content='12', name='multiply', tool_call_id='35f9612b-49da-4cee-a176-f8a7d3559893')

In [23]:
multiply.invoke({'name': 'multiply',
  'args': {'a': 3, 'b': 4},
  'id': '56cf2f39-9921-4dae-a461-d9a80508033e',
  'type': 'tool_call'})

ToolMessage(content='12', name='multiply', tool_call_id='56cf2f39-9921-4dae-a461-d9a80508033e')